# OBPS Data Processing

Processing OBPS data files into MESSAGE-ix format

In [1]:
# Read Excel files and import relevant packages

import pandas as pd
import os
import numpy as np
import re

# Sys and OS Operations
os.chdir("../../")




In [13]:
input_file = "data/MESSAGE_OBPS_update_(conservative_scenario).csv"
read_file = pd.read_csv(input_file, sep=',')

mapping = pd.read_excel("data/obps_inputs_clean.xlsx", sheet_name="Mapping")

#mapping["variable"] = mapping["variable"].str.split("|")[3:].aggregate(lambda x: "|".join(x))


# Data Viewer Cell

In [9]:
#mapping
read_file

,variable,region,time,value,unit [tCO2/-]
0,Input|Policy|OBPS|Standard|Aluminum|Alumina,Canada,2019,1.58,Tonnes of liquid aluminium
1,Input|Policy|OBPS|Standard|Aluminum|Alumina,Canada,2020,1.58,Tonnes of liquid aluminium
2,Input|Policy|OBPS|Standard|Aluminum|Alumina,Canada,2021,1.58,Tonnes of liquid aluminium
3,Input|Policy|OBPS|Standard|Aluminum|Alumina,Canada,2022,1.58,Tonnes of liquid aluminium
4,Input|Policy|OBPS|Standard|Aluminum|Alumina,Canada,2023,1.58,Tonnes of liquid aluminium
...,...,...,...,...,...
40656,Input|Policy|OBPS|Tightening|Wood Products|Woo...,Saskatchewan,2056,0.10,NaN
40657,Input|Policy|OBPS|Tightening|Wood Products|Woo...,Saskatchewan,2057,0.10,NaN
40658,Input|Policy|OBPS|Tightening|Wood Products|Woo...,Saskatchewan,2058,0.10,NaN
40659,Input|Policy|OBPS|Tightening|Wood Products|Woo...,Saskatchewan,2059,0.10,NaN


# Final Dataframe - Mapping Technologies

Created dictionaries for unit conversions

>Note: densities are recorded in tonnes per metre cubed

* Conversion of different commodity volumes to GJ
    1. For "oil", "loil", "ref_hil", and "foil" products: used heavy crude oil value
    2. For "oil_extr" products: used bitumen value (m3 to GJ)
    3. For "coal" and "igcc" products: used bitumenous coal value
    4. For "gas" products: assumed natural gas plant


In [ ]:
# Create dataframe that is a copy of the original dataframe, but with the corrsponding technologies in a "technology column"

#mapping_dict = dict(zip(mapping['variable'].astype(str), mapping['technology'].astype(str)))
#read_file['technology'] = read_file['variable'].astype(str).map(mapping_dict)
mapping_dict = mapping.groupby('variable')['technology'].apply(list).to_dict()
new_units_dict = dict(zip(read_file["unit [tCO2/-]"].unique(), ["tonne", "tonne", "MWh", "GWh", "tonne", "GWh", 
                                                                "CAN-CWB", "m3", "barrel_bitumen", "tonne_h2", 
                                                                "MWh", "m3", "AB_index", "100e3_m3", "1e3_AB-CWB", 
                                                                "CAN-CWB", "barrel_heavy_crude", "-"]))

#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#

# Iterate through the main dataframe and update 'technology' where a match is found
def tech_mapping(variable):
    techs = []
    for var, tech in mapping_dict.items():
        if var in variable:
            techs.extend(tech)

    try:
        return techs
    except Exception as e:
        return "No associated technologies"

def convert_to_KWa(data):

    # Conversion of different commodity volumes to GJ
        ## For "loil", "ref_hil", and "foil" products: used heavy crude oil value (m3 to GJ)
        ## For "oil_extr" products: used bitumen value (m3 to GJ)
        ## For "coal" and "igcc" products: used bitumenous coal value (tonnes to GJ)
        ## For "gas" products: assumed natural gas plant (m3 to GJ)
        
    # Densities in tonnes per m3
    general_conversion_dict = {"GJ_to_Kwa": 0.03171, "methanol_density": 0.7914, "barrel_to_m3": 0.159, 
                               "h2_density": 0.09e-3}
    
    # Conversion of different commodity volumes to GJ
    tech_to_GJ_dict = {"meth_": 15.6, "liq_": 23.6, "gas_": 0.0373, "loil_": 40.9,
                       "foil_": 40.9, "coal_": 27.6, "igcc": 27.6, "h2": 0.012, "ref_": 40.9,
                       "oil_extr": 42.8, "syn_liq": 38.51}
    
    # Initialize the converted_values list
    converted_values = []
    
    # Process "barrel_bitumen" conversion (vectorized)
    data.loc[data["unit [tCO2/-]"] == "barrel_bitumen", "value"] *= general_conversion_dict["barrel_to_m3"]
    
    # Process the technologies and apply the GJ conversion
    for tech, conversion_factor in tech_to_GJ_dict.items():
        mask = data["technology"].str.startswith(tech)
        data.loc[mask, "value"] *= conversion_factor
    
    # Final conversion to KWa
    data["conversion_to_KWa"] = data["value"]
    data["conversion_to_KWa"] *=  general_conversion_dict["GJ_to_Kwa"]
    
    # Return the updated "value" column
    return data["conversion_to_KWa"].tolist()
            
    

#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#


# Expand technology column with new rows for each technology
read_file["technology"] = read_file["variable"].apply(lambda x: tech_mapping(x))
read_file = read_file.explode("technology")
read_file = read_file[read_file["technology"].notna()]

read_file["unit [tCO2/-]"] = read_file["unit [tCO2/-]"].replace(new_units_dict)

read_file["conversion_to_KWa"] = convert_to_KWa(read_file)
read_file["comments"] = pd.NA # Add comments column



read_file.to_csv("data/New_OBPS.csv", index=False, header=True)

with pd.ExcelWriter("data/obps_inputs_clean.xlsx",engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    read_file.to_excel(writer, sheet_name="New_OBPS_Values")

In [43]:
#read_file["technology"].unique()
read_file["unit [tCO2/-]"].unique()

array(['tonne', 'tonnes of high-value chemicals from steam cracking',
       'megawatt hour', 'gigawatt hours (GWh)',
       'Mass (expressed in tonnes)',
       'Energy (expressed in gigawatt hours)',
       'Amount (expressed in CAN- CWB)', 'm3 of bitumen',
       'barrels of bitumen and heavy crude oil', 'Tonnes of hydrogen gas',
       'Megawatt hours (MWh)',
       'cubic metres of propane and butane combined at a temperature of 15°C and at an equilibrium pressure',
       'Alberta gas processing index',
       '100 000 cubic metres of pipeline-transmission- quality natural gas at a temperature of 15°C and a pressure of 101.325 kPa',
       'Alberta complexity weighted barrel (in thousand barrels per year)',
       'complexity-weighted barrels', 'barrels of synthetic crude oil',
       nan], dtype=object)

# Test Model

In [27]:
from model.Model import model_MSG

msg = model_MSG(scenario_name='baseline', config=False)
platform, msg_scenario, msg_folder = msg.setup_scenario()


Model Logs: Setting up scenario

Model Logs: Applying OBPS



KeyError: "['comments'] not found in axis"